# Image Processing in Python — lecture codeEvery snippet from *DIP-Python-Introduction-NDU.pptx*, made runnable and keyedto the slide it appears on. Run the cells in order during the lecture, or handthe notebook to students afterwards.Nothing to install in Colab. Locally:```pip install numpy opencv-python scikit-image matplotlib```

## Slide 8 — Getting set upThe opening cell. Worth typing from scratch in front of the class.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data

img = data.camera()

print(img.shape)          # (512, 512)
print(img.dtype)          # uint8
print(img.min(), img.max())

plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.axis('off')
plt.show()

## Slide 10 — An image is just a NumPy array

In [ ]:
print(img.shape)         # (512, 512)
print(img.dtype)         # uint8
print(img.ndim)          # 2  -> grayscale
print(img[213, 202])     # 43   a single pixel
print(img.mean())        # 129.06  one scalar over 262,144 pixels

# the 8 x 8 patch shown on the slide
print(img[210:218, 200:208])

## Slide 11 — dtype carries the range conventionMATLAB's class is a contract. A NumPy dtype is not.

In [ ]:
# im2double
f = img.astype(np.float64) / 255.0
print('float range :', f.min(), f.max())

# im2uint8
u = np.clip(f * 255, 0, 255).astype(np.uint8)
print('back to uint8:', u.dtype, u.min(), u.max())

# mat2gray - rescale an array whose range you do not know
x = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(f))))   # 0 .. ~11
print('unknown range:', x.min().round(2), x.max().round(2))

g = cv2.normalize(x, None, 0, 1, cv2.NORM_MINMAX, cv2.CV_64F)
print('after normalize:', g.min(), g.max())

## Slide 12 — Two indexing conventionsArrays are `(row, column)`. OpenCV points are `(x, y)` and sizes are `(width, height)`.

In [ ]:
y, x = 213, 202
print('img[y, x] =', img[y, x])          # row first

canvas = img.copy()
cv2.circle(canvas, (x, y), 25, 255, 2)   # x first
small = cv2.resize(img, (128, 64))       # width first
print('resize((128, 64)) gives shape', small.shape)   # (64, 128)

plt.imshow(canvas, cmap='gray', vmin=0, vmax=255); plt.axis('off'); plt.show()

## Slide 13 — Slicing replaces imcrop

In [ ]:
roi   = img[60:220, 180:340]
top   = img[:100, :]
right = img[:, -100:]
flip  = img[::-1, :]

for name, a in [('roi', roi), ('top', top), ('right', right), ('flip', flip)]:
    print(f'{name:6s} {a.shape}')

**Views versus copies.** The one real surprise. MATLAB always copies; NumPy does not.

In [ ]:
demo = img.copy()
view = demo[60:220, 180:340]     # a view into demo
view[:] = 0                      # writes straight through

safe = img[60:220, 180:340].copy()
safe[:] = 0                      # img is untouched

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(demo, cmap='gray', vmin=0, vmax=255); ax[0].set_title('after view[:] = 0')
ax[1].imshow(img,  cmap='gray', vmin=0, vmax=255); ax[1].set_title('after safe[:] = 0')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## Slide 15 — imread returns BGR, not RGBThere is no file to read here, so the BGR array `cv2.imread` would hand back isbuilt from a skimage RGB image. The point is identical.

In [ ]:
rgb_true = data.coffee()                     # skimage gives RGB
bgr      = cv2.cvtColor(rgb_true, cv2.COLOR_RGB2BGR)   # what imread returns

# bgr = cv2.imread('photo.jpg')              # the real call
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)   # the fix

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].imshow(bgr); ax[0].set_title('plt.imshow(bgr)  — wrong')
ax[1].imshow(rgb); ax[1].set_title('after cvtColor  — right')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

# grayscale sidesteps the whole issue
# g = cv2.imread('photo.jpg', cv2.IMREAD_GRAYSCALE)

## Slide 16 — The display helperPaste this into the first cell of every notebook this term.

In [ ]:
def show(*imgs, titles=None, size=4):
    n = len(imgs)
    fig, ax = plt.subplots(1, n, figsize=(size*n, size))
    ax = np.atleast_1d(ax).ravel()
    for i, (a, im) in enumerate(zip(ax, imgs)):
        if im.ndim == 2:
            a.imshow(im, cmap='gray', vmin=0, vmax=255)
        else:
            a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        if titles:
            a.set_title(titles[i])
        a.axis('off')
    plt.tight_layout()
    plt.show()


show(img,
     cv2.GaussianBlur(img, (9, 9), 0),
     cv2.Canny(img, 100, 200),
     titles=['original', 'Gaussian 9x9', 'Canny'], size=3)

## Slide 17 — Always pin vmin and vmax

In [ ]:
dark = (img * 0.35).astype(np.uint8)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(dark, cmap='gray')
ax[0].set_title("cmap='gray'  — autoscaled, lies")
ax[1].imshow(dark, cmap='gray', vmin=0, vmax=255)
ax[1].set_title('vmin=0, vmax=255  — true')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

print('the array really spans', dark.min(), 'to', dark.max())

## Slide 19 — Point operations are array arithmetic

In [ ]:
neg = 255 - img                                 # imcomplement
bw  = ((img > 128) * 255).astype(np.uint8)      # im2bw

f   = img / 255.0                               # gamma / imadjust
out = ((f ** 0.4) * 255).astype(np.uint8)

show(img, neg, bw, out,
     titles=['original', '255 - img', 'img > 128', 'gamma 0.4'], size=3)

## Slide 20 — Integer arithmetic wrapsAsk the room what `200 + 100` gives before running this.

In [ ]:
p = np.uint8([[200]])                  # a one-pixel image

print('p + 100          =', (p + np.uint8(100))[0, 0])   # 44   wraps
print('cv2.add(p, 100)  =', cv2.add(p, 100)[0, 0])       # 255  saturates

widened = np.clip(p.astype(np.int16) + 100, 0, 255).astype(np.uint8)
print('widen and clip   =', widened[0, 0])

# the same thing across the whole intensity range
r = np.arange(256, dtype=np.uint8)
plt.plot(r, r + np.uint8(100), label='img + 100  (wraps)')
plt.plot(r, cv2.add(r.reshape(-1, 1), 100).ravel(), label='cv2.add  (saturates)')
plt.xlabel('input'); plt.ylabel('output'); plt.legend(frameon=False); plt.show()

## Slide 21 — Lookup tablesEvery point transformation in the course should end up here.

In [ ]:
moon = data.moon()

r   = np.arange(256)
lut = np.clip((r - 80) * 255.0 / 80, 0, 255).astype(np.uint8)
out = cv2.LUT(moon, lut)

fig = plt.figure(figsize=(11, 3.4))
a = fig.add_subplot(1, 3, 1); a.plot(lut); a.set_xlim(0, 255); a.set_ylim(0, 255)
a.set_xlabel('r'); a.set_ylabel('s'); a.set_title('the lookup table')
for k, (im, t) in enumerate([(moon, 'before'), (out, 'after cv2.LUT')]):
    b = fig.add_subplot(1, 3, k+2)
    b.imshow(im, cmap='gray', vmin=0, vmax=255); b.set_title(t); b.axis('off')
plt.tight_layout(); plt.show()

## Slide 22 — Histograms

In [ ]:
h1 = np.bincount(moon.ravel(), minlength=256)              # imhist
h2 = cv2.calcHist([moon], [0], None, [256], [0, 256])      # OpenCV

print('identical:', np.allclose(h1, h2.ravel()))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].imshow(moon, cmap='gray', vmin=0, vmax=255); ax[0].axis('off'); ax[0].set_title('moon()')
ax[1].bar(np.arange(256), h1, width=1.0); ax[1].set_xlim(0, 255)
ax[1].set_title('bincount — full 0..255 range')
ax[2].hist(moon.ravel())                                    # MATLAB's hist
ax[2].set_title('plt.hist — data range, 10 bins')
plt.tight_layout(); plt.show()

## Slide 24 — MATLAB to Python| MATLAB | Python ||---|---|| `imread` / `imwrite` | `cv2.imread` / `cv2.imwrite` || `im2double` | `img.astype(np.float64) / 255` || `mat2gray` | `cv2.normalize(..., cv2.NORM_MINMAX)` || `imcomplement` | `255 - img` || `imcrop` | `img[r0:r1, c0:c1]` || `imhist` | `np.bincount(img.ravel(), minlength=256)` || `histeq` | `cv2.equalizeHist` || `adapthisteq` | `cv2.createCLAHE` || `imfilter` | `cv2.filter2D` || `graythresh` | `cv2.threshold(..., cv2.THRESH_OTSU)` || `intlut` | `cv2.LUT` || `regionprops` | `skimage.measure.regionprops` |## Slide 25 — The five that cost the most time1. **BGR, not RGB** — `cvtColor` on display, always.2. **Integer arithmetic wraps** — `cv2.add`, or widen and clip.3. **Matplotlib autoscales** — pin `vmin=0, vmax=255` on every uint8 image.4. **dtype does not imply a range** — print it and check.5. **Slices are views** — writing into a region of interest writes into the original.